<a href="https://colab.research.google.com/github/Alle84fr/dio_audio_autom/blob/main/audio_rio_ia.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

depois imports

(faz parte do Ipython/Jupyter, mostra imagens, audios, videos, html, sgv)
Ipython.disply -> audio, display, javascript

(módulo dentro do google colab que permite acesso a uploads, downloads, conexões, configurações de ambientes)
google.colab -> output

(módulo de biblioteca do py, que serve para codificar e decodificar dados em base65, transforma buinários em textos seguros para transmissão - envia em Json, incorporad dados em html)
base64 -> base64code

In [ ]:
language = "pt"

In [ ]:
from IPython.display import Javascript, Audio
from google.colab import output
from base64 import b64decode
from io import BytesIO
!pip -q install pydub
from pydub import AudioSegment

Código js pré estabelecido que grava áudio do microfone
é funmcional e, qualquer browser e conversa com .py

só roda String REVORD

In [ ]:

RECORD = """
const sleep  = time => new Promise(resolve => setTimeout(resolve, time))
const b2text = blob => new Promise(resolve => {
  const reader = new FileReader()
  reader.onloadend = e => resolve(e.srcElement.result)
  reader.readAsDataURL(blob)
})
var record = time => new Promise(async resolve => {
  stream = await navigator.mediaDevices.getUserMedia({ audio: true })
  recorder = new MediaRecorder(stream)
  chunks = []
  recorder.ondataavailable = e => chunks.push(e.data)
  recorder.start()
  await sleep(time)
  recorder.onstop = async ()=>{
    blob = new Blob(chunks)
    text = await b2text(blob)
    resolve(text)
  }
  recorder.stop()
})
"""

In [ ]:
#para gravar, com 5 seg
def record(sec = 5):

    #dispçay no cod js
    display(Javascript(RECORD))

    #resposta do display, do .eval a ...é para executar var record do js
    # parâmetro %s está relacionado com tempo sec =5 (* mil porque no record do js é milesegundos)
    js_result = output.eval_js('record(%s)' % (sec * 1000))

    #pegar audio e decodifica com b64
    # como vem com alguns seguimento, preciso de um parte, trazer split
    audio = b64decode(js_result.split(",")[1])

    # a = AudioSegment.from_file(BytesIO(audio))

    with open("audio.wav", "wb") as f:
      f.write(audio)

    return "audio.wav"


In [59]:
print("TEstando record, 1 2 3")
record_file = record()
display(Audio(record_file, autoplay=True))

TEstando record, 1 2 3


<IPython.core.display.Javascript object>

observação

KeyboardInterrupt                         Traceback (most recent call last)
/tmp/ipykernel_6051/739318327.py in <cell line: 0>()
----> 1 record_file = record()
      2 display(Audio(record_file, autoplay=True))

11 frames
/usr/local/lib/python3.12/dist-packages/zmq/error.py in __init__(self, errno, msg)
     45             self.errno = errno
     46             if msg is None:
---> 47                 self.strerror = strerror(errno)
     48             else:
     49                 self.strerror = msg

KeyboardInterrupt:

erro de permissão do microfone


# Nova secção

descrição

In [60]:
!pip install git+https://github.com/openai/whisper.git -q

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [61]:

import whisper

In [62]:
model = whisper.load_model("small")

#transcrever
#fp = floating, numero para calculo de ia
# 16 é para ser mais rápido
# neste caso como é False, usa-se 32 que é mais preciso
result = model.transcribe(record_file, fp16=False)

In [63]:
transcription = result["text"]
print(transcription)

# se saída for vazia provavelmente é permissão de uso do microfone ou configuração do microfone no computador

 Abra a caraba.


Para encontrar o aquivo ir na pasta ao lado (ficheiro)


integração

In [ ]:
!pip install openai

In [ ]:
import os
os.environ["OPENAI_API_KEY"] = "COLOQUE_SUA_CHAVE_AQUI"

In [ ]:
import openai


In [ ]:
# api_key - site openai, gerar chave
openai.api_key = os.getenv("OPENAI_API_KEY")


In [66]:
try:
    # chamada para o ChatGPT
    response = openai.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[{"role": "user", "content": transcription}]
    )

    # pegar a resposta
    chatgpt_response = response.choices[0].message.content
    print("ChatGPT respondeu:", chatgpt_response)

except openai.error.RateLimitError:
    print("⚠️ Limite da API atingido. Verifique seu crédito na OpenAI.")
except Exception as e:
    print("❌ Ocorreu um erro inesperado:", str(e))

AttributeError: module 'openai' has no attribute 'error'

sintetiza com voz

texto em voz

In [1]:
!pip install gtts

In [2]:
from gtts import  gTTS

In [ ]:
gtts_object = gTTS(text=chatgpt_response, lang=language, slow=False)

response_audio = "/content/response_audio.wav"

gtts_object.save(response_audio)

display(Audio(response_audio, autoplay=True))